# Pipelines

A `Pipeline` wires functions and stream operators into a reusable computation you define
once and run either on stored arrays or live, event by event, with identical
results. It is the capstone of screamer's batch-equals-streaming property: the
guarantee that holds for a single function holds for a whole graph of them.

This notebook builds one small graph, a smoothed spread over two price feeds, and
uses it to show everything a `Pipeline` does: inspect its structure, run it in batch
and live, expose several outputs, and save and reload it as JSON. The concepts
behind the graph are in [Pipelines](../pipelines.md).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from screamer import Input, Pipeline, RollingMean, Sub
from screamer import CombineLatest

## Building the graph

`Input(name)` creates a named source. Applying stream operators and functions to
those nodes builds more nodes: `CombineLatest` aligns the two feeds into one
two-column stream, `Sub()` takes their difference, and `RollingMean(20)` smooths
it. Nothing computes yet; building the graph only records structure. Compiling it
into a `Pipeline` names which nodes are inputs and which are outputs.

In [ ]:
a, b = Input("a"), Input("b")

aligned = CombineLatest()(a, b)    # align the two feeds -> a node
spread  = Sub()(aligned)           # difference          -> a node
signal  = RollingMean(20)(spread)  # smooth              -> a node

pipe = Pipeline(inputs=[a, b], outputs=[signal])

## Inspecting the graph

`print(pipe)` shows the graph as an indented tree, rooted at each output and descending to the inputs, with each node's parameters in its label. This requires no optional dependencies, so it works in any environment. Each node also carries an id (`#2`); a node shared by several consumers is printed once and then referenced by that id.

In [ ]:
print(pipe)

The same graph is available as a Graphviz DOT string via `pipe.to_dot()`, which you
can feed to any DOT renderer.

In [ ]:
print(pipe.to_dot())

In a notebook, displaying the `pipe` object directly renders that diagram inline.

In [ ]:
pipe.to_graphviz()

## Running it: batch and live

`pipe(*feeds)` runs the whole graph over stored arrays in one pass, the batch path. Pass generators of `(value, index)` pairs instead of arrays to run event by event: `pipe(gen_a, gen_b)` returns an iterator of `(value, index)` events numerically identical to the batch result. Each feed is a `(values, index)` pair.

In [ ]:
rng = np.random.default_rng(0)
n = 200
index = np.arange(1, n + 1, dtype=np.int64)
fa = (100 + np.cumsum(rng.standard_normal(n)), index)   # two feeds on the
fb = (100 + np.cumsum(rng.standard_normal(n)), index)   # same timeline

batch_v, batch_idx = pipe(fa, fb)          # whole arrays in

# Event by event via generators (byte-identical to batch)
gen_a = ((v, k) for v, k in zip(fa[0], fa[1]))
gen_b = ((v, k) for v, k in zip(fb[0], fb[1]))
events = list(pipe(gen_a, gen_b))
live_v   = np.array([e[0] for e in events])
live_idx = np.array([e[1] for e in events])

print("batch and live match:",
      np.array_equal(batch_idx, live_idx) and np.array_equal(batch_v, live_v, equal_nan=True))

import pandas as pd
m = ~np.isnan(batch_v)
pd.DataFrame({"batch": batch_v[m][:5], "live": live_v[m][:5]},
             index=batch_idx[m][:5]).rename_axis("index")

## Multiple outputs

A graph can expose more than one output. Adding `spread` alongside `signal` gives
back the raw difference and its smoothed version together, each as its own
`(values, index)` pair. With `align_outputs=True` (the default), all outputs are
co-indexed onto a shared sorted axis, so every pair has the same length and
outputs can be compared sample for sample. Set `align_outputs=False` to get
independent per-output streams whose lengths may differ.

In [ ]:
dag2 = Pipeline(inputs=[a, b], outputs=[spread, signal])
(raw_v, raw_idx), (sig_v, sig_idx) = dag2(fa, fb)

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(raw_idx, raw_v, lw=0.6, color="0.7", label="spread (raw)")
ax.plot(sig_idx, sig_v, lw=1.6, color="crimson", label="RollingMean(20)")
ax.axhline(0, color="k", lw=0.4)
ax.set_xlabel("index")
ax.set_ylabel("a - b")
ax.legend(loc="upper left")
plt.tight_layout()

## Saving and reloading as JSON

`pipe.to_json()` serializes the whole graph, input names, every node with its
parameters, the outputs, and `align_outputs`, into readable JSON you can write to
a config file. `Pipeline.from_json(text)` rebuilds a runnable graph that produces
identical results, so a saved graph can be reloaded later exactly as it was.
(`to_dict` / `from_dict` do the same with a plain dict.)

In [ ]:
import json

config = pipe.to_json()
print(json.dumps(json.loads(config), indent=2))

restored = Pipeline.from_json(config)
restored_v, _ = restored(fa, fb)
print("\nrestored graph matches the original:",
      np.array_equal(batch_v, restored_v, equal_nan=True))